In [1]:
# Top cell: install the libraries this notebook uses.
# If they are already installed, pip will simply say so.
%pip install -q pandas matplotlib numpy scipy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path
ROOT = Path('..').resolve()
PROC = ROOT / 'data' / 'processed'
CH = ROOT / 'charts'
CH.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi': 150})
print('Paths ready:', PROC.exists(), CH.exists())

Paths ready: True True


# 3. Extreme events: frequency and costs over time

Plain words first. This chapter uses the United States only, because that is what the source covers. A billion-dollar disaster is a weather or climate event whose total direct losses reached at least 1 billion US dollars. Costs here are CPI adjusted, which means they are adjusted for inflation so that a dollar in 1980 and a dollar in 2024 are compared on a fairer basis. CPI stands for Consumer Price Index.

Important limit: this dataset counts only events that crossed the 1 billion dollar line. It does not count every flood, storm, or fire. So a rise in this count can reflect more large events, more valuable property in harm's way, better reporting, or a mix of those. The dataset alone cannot split those causes.

Source: NOAA NCEI, Billion-Dollar Disasters archive, 1980 to 2024, 403 events. NOAA stopped updating this product in 2025. Climate Central now continues a similar list, but its post 2024 events are not included here, so the years stay comparable.

In [3]:
ev = pd.read_csv(PROC / 'disasters_events.csv')
print('Events:', len(ev))
print('Total CPI adjusted cost, billion USD:', round(ev['cost_cpi_adjusted_billion_usd'].sum(), 1))
print('Total deaths in file:', int(ev['deaths'].sum()))
assert len(ev) == 403, 'expected 403 events'
print(ev.head().to_string(index=False))

Events: 403
Total CPI adjusted cost, billion USD: 2917.6
Total deaths in file: 16941
                                                   name    disaster_type  begin_date  end_date  year  cost_cpi_adjusted_billion_usd  cost_unadjusted_billion_usd  deaths
       Southern Severe Storms and Flooding (April 1980)         Flooding    19800410  19800417  1980                         2.7564                       0.7068      30
                          Hurricane Allen (August 1980) Tropical Cyclone    19800807  19800811  1980                         2.2362                       0.5900      13
   Central/Eastern Drought/Heat Wave (Summer-Fall 1980)          Drought    19800601  19801130  1980                        40.6812                      10.0200    1260
                          Florida Freeze (January 1981)           Freeze    19810112  19810114  1981                         2.0764                       0.5720       0
Severe Storms, Flash Floods, Hail, Tornadoes (May 1981)     Severe Sto

In [4]:
by = pd.read_csv(PROC / 'disasters_by_year.csv')
print(by.tail(10).to_string(index=False))
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(by['year'], by['event_count'], color='#ff7f0e')
ax.set_title('Billion-dollar disasters in the United States, count by year, 1980 to 2024')
ax.set_xlabel('Year')
ax.set_ylabel('Number of events')
fig.savefig(CH / '08_disasters_count_by_year.png', bbox_inches='tight')
plt.show()

 year  event_count  total_cost_cpi_billion_usd  total_deaths  mean_cost_per_event_billion_usd
 2015           11                     30.9094           158                         2.809945
 2016           15                     61.2099           138                         4.080660
 2017           19                    395.9362          3280                        20.838747
 2018           16                    116.0339           247                         7.252119
 2019           14                     55.5155            44                         3.965393
 2020           22                    120.5847           262                         5.481123
 2021           20                    164.2988           724                         8.214940
 2022           19                    188.4634           496                         9.919126
 2023           27                     90.4686           470                         3.350689
 2024           27                    182.7136           568

In [5]:
bd = pd.read_csv(PROC / 'disasters_by_decade.csv')
print(bd.to_string(index=False))
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(bd['decade'], bd['total_cost_cpi_billion_usd'], color='#d62728')
ax.set_title('Total CPI adjusted cost of billion-dollar disasters, by decade')
ax.set_xlabel('Decade (2020s covers 2020-2024 only)')
ax.set_ylabel('Billion US dollars, CPI adjusted')
plt.setp(ax.get_xticklabels(), rotation=20, ha='right')
fig.savefig(CH / '09_disasters_cost_by_decade.png', bbox_inches='tight')
plt.show()

           decade  event_count  total_cost_cpi_billion_usd  total_deaths  mean_cost_per_event_billion_usd
            1980s           33                    219.6485          3017                         6.656015
            1990s           57                    335.1072          3075                         5.879074
            2000s           67                    621.9355          3102                         9.282619
            2010s          131                    994.3862          5227                         7.590734
2020s (2020-2024)          115                    746.5291          2520                         6.491557


In [6]:
by_type = ev.groupby('disaster_type').agg(count=('name', 'count'), cost=('cost_cpi_adjusted_billion_usd', 'sum')).sort_values('count', ascending=False)
print(by_type.round(1).to_string())
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(by_type.index, by_type['count'], color='#2ca02c')
ax.set_title('Billion-dollar disasters by type, 1980 to 2024: count')
ax.set_xlabel('Disaster type')
ax.set_ylabel('Number of events')
plt.setp(ax.get_xticklabels(), rotation=20, ha='right')
fig.savefig(CH / '10_disasters_by_type_count.png', bbox_inches='tight')
plt.show()

                  count    cost
disaster_type                  
Severe Storm        203   514.0
Tropical Cyclone     67  1543.0
Flooding             45   203.0
Drought              32   367.8
Winter Storm         24   104.4
Wildfire             23   148.0
Freeze                9    37.3


In [7]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(by_type.sort_values('cost', ascending=False).index, by_type.sort_values('cost', ascending=False)['cost'], color='#9467bd')
ax.set_title('Billion-dollar disasters by type, 1980 to 2024: total cost')
ax.set_xlabel('Disaster type')
ax.set_ylabel('Billion US dollars, CPI adjusted')
plt.setp(ax.get_xticklabels(), rotation=20, ha='right')
fig.savefig(CH / '11_disasters_by_type_cost.png', bbox_inches='tight')
plt.show()
print('Costliest single event:')
print(ev.sort_values('cost_cpi_adjusted_billion_usd', ascending=False).head(3)[['name', 'disaster_type', 'year', 'cost_cpi_adjusted_billion_usd', 'deaths']].to_string(index=False))

Costliest single event:
                           name    disaster_type  year  cost_cpi_adjusted_billion_usd  deaths
Hurricane Katrina (August 2005) Tropical Cyclone  2005                       201.2975    1833
 Hurricane Harvey (August 2017) Tropical Cyclone  2017                       160.0000      89
 Hurricane Ian (September 2022) Tropical Cyclone  2022                       119.6260     152


## What this shows

Severe storms are the most frequent type, with 203 of 403 events, but tropical cyclones cost the most in total, about 1,543 billion dollars CPI adjusted, from only 67 events. The single costliest event in the file is Hurricane Katrina in 2005, at about 201.3 billion dollars CPI adjusted. The deadliest is Hurricane Maria in 2017, with 2,981 deaths listed. Counts rose from 33 events in the 1980s to 131 in the 2010s, and 115 more in just 2020 to 2024. Total cost rose too, but as notebook 5 shows, the average cost per event did not rise in the same way. That split is the key story: more events, not simply more expensive events.